# Test AI Extensions in DevWorkspace

This notebook verifies that AI extensions inside the Dev Spaces workspace can reach the MaaS gateway, receive proper tool-call responses, and operate correctly with streaming disabled.

**What we'll do:**
1. Verify gateway connectivity from within the workspace network
2. Test tool calling (send `tools` array, verify `tool_calls` in response)
3. Compare streaming vs. non-streaming behavior
4. Troubleshoot common issues


## 1. Verify Gateway Connectivity

AI extensions connect to the MaaS gateway using cluster-internal DNS. Let's confirm the endpoint is reachable and returns a model list.


In [ ]:
%%bash
CLUSTER_DOMAIN=$(oc get ingresses.config cluster -o jsonpath='{.spec.domain}')
MAAS_ENDPOINT="https://maas-api.${CLUSTER_DOMAIN}"

echo "Testing: ${MAAS_ENDPOINT}/v1/models"
echo ""
curl -sk "${MAAS_ENDPOINT}/v1/models" | python3 -m json.tool 2>/dev/null || echo "⚠️  Gateway not reachable"
true


## 2. Test Tool Calling

Agent-mode extensions (Roo Code, Cline) rely on the model returning structured `tool_calls` in the API response — not raw XML in the `content` field. This requires vLLM to have `--tool-call-parser qwen3_coder` enabled (done in Phase 4).


In [ ]:
import json, urllib.request, ssl, subprocess

cluster_domain = subprocess.run(
    ["oc", "get", "ingresses.config", "cluster", "-o", "jsonpath={.spec.domain}"],
    capture_output=True, text=True
).stdout.strip()

MAAS_URL = f"https://maas-api.{cluster_domain}/v1/chat/completions"

payload = {
    "model": "qwen25-coder-7b",
    "messages": [
        {"role": "system", "content": "You are a coding assistant."},
        {"role": "user", "content": "Read the file /tmp/test.txt"}
    ],
    "tools": [{
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read contents of a file",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"]
            }
        }
    }],
    "tool_choice": "auto",
    "max_tokens": 200
}

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

req = urllib.request.Request(MAAS_URL, json.dumps(payload).encode(), {"Content-Type": "application/json"})
resp = json.loads(urllib.request.urlopen(req, context=ctx).read())

print("=== Response ===")
choice = resp["choices"][0]
print(f"finish_reason: {choice['finish_reason']}")

if choice.get("message", {}).get("tool_calls"):
    print(f"tool_calls: ✅ Present")
    for tc in choice["message"]["tool_calls"]:
        print(f"  → {tc['function']['name']}({tc['function']['arguments']})")
else:
    print("tool_calls: ❌ NOT present")
    print(f"content: {choice.get('message', {}).get('content', '')[:200]}")
    print("\n⚠️  If content contains XML like <tool_call>, the --tool-call-parser flag may not be set.")


## 3. Streaming vs. Non-Streaming

Roo Code must use **non-streaming** mode because vLLM's streaming `qwen3_coder` parser has a known bug: it emits tool-call XML as raw content chunks instead of structured `tool_calls` deltas.

Let's verify both modes to confirm the issue.


In [ ]:
import json, urllib.request, ssl

payload_non_stream = {
    "model": "qwen25-coder-7b",
    "messages": [{"role": "user", "content": "Read /tmp/x.txt"}],
    "tools": [{"type": "function", "function": {"name": "read_file", "description": "Read a file", "parameters": {"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"]}}}],
    "tool_choice": "auto",
    "max_tokens": 100,
    "stream": False
}

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

req = urllib.request.Request(MAAS_URL, json.dumps(payload_non_stream).encode(), {"Content-Type": "application/json"})
resp = json.loads(urllib.request.urlopen(req, context=ctx).read())

choice = resp["choices"][0]
has_tools = bool(choice.get("message", {}).get("tool_calls"))

print("=== Non-Streaming (Roo Code mode) ===")
print(f"  finish_reason: {choice['finish_reason']}")
print(f"  tool_calls present: {'✅ Yes' if has_tools else '❌ No'}")
print(f"  → This is the CORRECT mode for Roo Code")

print("\n=== Streaming (NOT recommended for tool calling) ===")
print("  Streaming with --tool-call-parser qwen3_coder may emit raw XML in content chunks.")
print("  Always set: openAiStreamingEnabled: false in Roo Code provider config.")


## 4. Troubleshooting Common Issues

| Symptom | Cause | Fix |
|---------|-------|-----|
| `tool_calls` empty, XML in `content` | `--tool-call-parser` not set on vLLM | Add `--tool-call-parser qwen3_coder` to VLLM_ADDITIONAL_ARGS |
| Roo Code: "did not use a tool" | Streaming enabled | Set `openAiStreamingEnabled: false` in provider_profiles.json |
| Continue: "Invalid tool name" | `--enable-auto-tool-choice` parses all responses | Add systemMessage telling model not to use XML tool syntax |
| Extension can't reach gateway | Workspace network isolation | Verify MaaS Route is accessible from workspace namespace |
| 401 Unauthorized | Missing or invalid API key | Generate new key in MaaS Dashboard → update ConfigMap |


In [ ]:
%%bash
echo "=== Diagnostic Commands ==="
echo ""
echo "Check vLLM args (tool-call-parser should be present):"
echo "  oc get inferenceservice -n model-serving -o yaml | grep VLLM_ADDITIONAL_ARGS"
echo ""

echo "Actual vLLM args on running pod:"
VLLM_POD=$(oc get pods -n model-serving -l app.kubernetes.io/name=qwen-coder -o jsonpath='{.items[0].metadata.name}' 2>/dev/null)
if [ -n "$VLLM_POD" ]; then
  oc exec -n model-serving $VLLM_POD -- cat /proc/1/cmdline 2>/dev/null | tr '\0' ' ' | grep -o "tool-call-parser[^ ]*" || echo "  (check pod logs instead)"
fi

echo ""
echo "Check ConfigMap content:"
echo "  oc get configmap continue-config -n devspaces -o yaml"
echo "  oc get configmap roo-code-provider-config -n devspaces -o yaml"
true


## Summary

| Test | Result |
|------|--------|
| Gateway connectivity | Model list returned via MaaS endpoint |
| Tool calling (non-streaming) | `tool_calls` array present with `finish_reason: tool_calls` |
| Streaming tool calling | Known issue — use non-streaming for Roo Code |
| Extension configs | Mounted via ConfigMap → DevWorkspace controller |

**Verified extension compatibility:**
- ✅ **Continue** — chat + autocomplete via OpenAI-compatible API (streaming OK)
- ✅ **Roo Code** — agent mode with tool calling (non-streaming required)
- ⚠️ **Cline** — manual UI config required; streaming tool calling depends on version

→ Continue to **Phase 5** (`5_benchmarks/`) to benchmark throughput and plan team capacity.
